# Day 6 & 7: Post-Training Evaluation and HF Push

Goal: Run full before/after evaluation. Push adapter to HF Hub.

In [ ]:
!pip install -q datasets transformers peft trl bitsandbytes accelerate wandb evaluate rouge_score -U

In [ ]:
import json
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import HfApi

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    "outputs/meditune-checkpoint"  # Update this path if different
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Evaluate
# ft_results = evaluate_pubmedqa(finetuned_model, tokenizer, n_samples=500)
# ... see src/evaluate.py for the evaluation code ...


In [ ]:
# Merge LoRA weights into base model (if memory permits, else push adapter only)
print("Merging LoRA adapter...")
# merged_model = finetuned_model.merge_and_unload()

# Use kaggle secrets to get HUGGINGFACE_TOKEN
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
# finetuned_model.push_to_hub("nikhilsh10/meditune-mistral-7b", use_auth_token=token)